# 02 — Frozen multilingual comparison and selection
Inputs: notebook 01's fixed data and pinned training dependencies. Cap: 15 minutes including encoder setup.
Output: `runs/final` frozen selected bundle, development metrics and export parity.
Skip E5 if download/setup exceeds budget; record fallback. Never fine-tune the encoder here.
Use the same local/Drive setup as notebook 01. No variables from notebook 01 are required.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json
# Local: run from repo root or training/notebooks. Colab: copy private project to /content/svo.
ROOT = Path('/content/svo') if Path('/content/svo/training').exists() else Path.cwd()
while not (ROOT / 'training').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'training/src/svo').exists(), 'Set ROOT to the extracted project root'
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT / 'training/src')
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'
PYTHON = str(ROOT / 'training/.venv/bin/python') if (ROOT / 'training/.venv/bin/python').exists() else sys.executable
DATA = ROOT / 'training/data/processed/messages.jsonl'
DRIVE_ROOT = Path('/content/drive/MyDrive/ScamVariantObservatory')
def run(*args):
    subprocess.run([PYTHON, *map(str,args)], check=True, cwd=ROOT, env=os.environ.copy())
run('-c', 'import numpy, sklearn, joblib; print("Core dependencies ready")')


In [ ]:
ENCODER = ROOT / 'training/cache/multilingual-e5-small'
TRY_E5 = True
if TRY_E5 and not (ENCODER / 'revision.json').exists():
    try:
        subprocess.run([PYTHON, 'training/scripts/download_encoder.py'], cwd=ROOT, check=True, timeout=600)
    except (subprocess.TimeoutExpired, subprocess.CalledProcessError):
        print('Encoder setup failed or exceeded budget; use TF-IDF fallback.')
RUN = ROOT / 'training/runs/final'
args = ['-m','svo.train','--data',DATA,'--run',RUN,'--simulation']
if TRY_E5 and (ENCODER / 'revision.json').exists():
    args += ['--encoder', ENCODER]
if not (RUN / 'freeze.json').exists():
    run(*args)
run('training/scripts/verify_bundle.py', RUN / 'bundle')
print(json.loads((RUN / 'development.json').read_text()))


In [ ]:
# Optional Colab checkpoint. Mount Drive before setting this flag.
COPY_TO_DRIVE = False
if COPY_TO_DRIVE:
    import shutil
    assert DRIVE_ROOT.parent.exists(), 'Mount Drive first'
    destination = DRIVE_ROOT / 'training-checkpoints' / RUN.name
    shutil.copytree(RUN, destination, dirs_exist_ok=False)
    print('Checkpoint copied; treat it as private.')
print('Local stage artifacts:', RUN)
